<a href="https://colab.research.google.com/github/wulandarifauziah/chatbot_hacktiv8_vol2/blob/main/CyberMateAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install -q streamlit pyngrok groq
!fuser -k 8501/tcp 2>/dev/null || true

In [35]:
from pyngrok import ngrok
from google.colab import userdata

try:
    token = userdata.get("NGROK_TOKEN")
    ngrok.set_auth_token(token)
    print("Token ngrok berhasil dikonfigurasi.")
except Exception as e:
    if "already set in configuration file" in str(e):
        print("Token ngrok sudah tersimpan. Kita lanjut.")
    else:
        print("Periksa secret NGROK_TOKEN di menu Secrets Colab.")
        print(type(e).__name__, str(e))

Token ngrok sudah tersimpan. Kita lanjut.


In [36]:

%%writefile streamlit_chat_app.py
import os
import streamlit as st
from groq import Groq

st.set_page_config(
    page_title="CyberMate AI",
    page_icon="🛡️",
    layout="centered"
)

st.title("🛡️ CyberMate AI")
st.caption(
    "Asisten AI untuk belajar keamanan siber "
    "dengan bahasa Indonesia yang mudah dipahami."
)

SYSTEM_PROMPT = """
Kamu adalah CyberMate AI, asisten edukasi keamanan siber.
Gunakan bahasa Indonesia yang ramah dan mudah dipahami.
Jelaskan phishing, malware, password, privasi digital,
dan keamanan jaringan dengan contoh sederhana.
Berikan langkah pencegahan praktis.
Utamakan keamanan dan edukasi.
Jangan memberi panduan untuk menyerang sistem tanpa izin.
Jika tidak yakin, katakan dengan jujur.
"""

if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    st.subheader("⚙️ Pengaturan Chatbot")

    temperature = st.slider(
        "Kreativitas jawaban",
        0.0, 1.0, 0.5, 0.1
    )

    max_tokens = st.selectbox(
        "Panjang jawaban",
        [250, 500, 800],
        index=1
    )

    if st.button("Reset Percakapan"):
        st.session_state.messages = []
        st.rerun()

# API key dibaca dari environment, bukan dari input pengguna
groq_api_key = os.environ.get("GROQ_API_KEY")

if not groq_api_key:
    st.error(
        "API belum dikonfigurasi oleh pemilik aplikasi. "
        "Silakan coba lagi nanti."
    )
    st.stop()

client = Groq(api_key=groq_api_key)

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

prompt = st.chat_input(
    "Tanyakan sesuatu tentang keamanan siber..."
)

if prompt:
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message("user"):
        st.markdown(prompt)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + st.session_state.messages

    try:
        with st.chat_message("assistant"):
            with st.spinner("CyberMate sedang berpikir..."):
                response = client.chat.completions.create(
                    model="openai/gpt-oss-20b",
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens
                )

                answer = (
                    response.choices[0].message.content
                    or "Maaf, AI belum menghasilkan jawaban."
                )
                st.markdown(answer)

        st.session_state.messages.append({
            "role": "assistant",
            "content": answer
        })

    except Exception as e:
        st.session_state.messages.pop()
        st.error("Permintaan ke AI gagal. Silakan coba lagi.")
        print(f"Groq API error: {type(e).__name__}: {e}")


Overwriting streamlit_chat_app.py


In [37]:

import os
import time
import subprocess
from pathlib import Path
from google.colab import userdata

# Ambil API key dari Colab Secrets
groq_key = userdata.get("GROQ_API_KEY")

# Siapkan environment aplikasi
env = os.environ.copy()
env["GROQ_API_KEY"] = groq_key

# Jalankan Streamlit dan simpan log error
log_path = "/tmp/cybermate_streamlit.log"
log_file = open(log_path, "w")

proc = subprocess.Popen(
    [
        "streamlit", "run", "streamlit_chat_app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

if proc.poll() is not None:
    log_file.close()
    print("Streamlit berhenti. Ini penyebab yang tercatat:")
    print(Path(log_path).read_text()[-6000:])
else:
    print("Streamlit masih berjalan.")
    print("Proses ID:", proc.pid)


Streamlit masih berjalan.
Proses ID: 13776


In [38]:
from pyngrok import ngrok

tunnel = ngrok.connect(8501)
print("URL CyberMate AI:", tunnel.public_url)

URL CyberMate AI: https://reboot-twenty-causal.ngrok-free.dev
